# FuncADL ServiceX Tutorial

## Background

### Prequisites
This tutorial assumes that you have completed the [ServiceX 15 Minute Histogram Challenge](https://tryservicex.org/challenge/) and the [User Guide](https://tryservicex.org/guide/).
### Introduction
ServiceX supports multiple query backends to suit different workflows. This tutorial covers what is FuncADL, when to use it, and how to build ServiceX queries with it.
This tutorial is based on the content that can be found in the [FuncADL user guide](https://tryservicex.org/funcadl/).
### What is FuncADL
FuncADL is an Analysis Description Language inspired by functional languages and C#’s LINQ. Sophisticated filtering and computation of new values can be expressed by chaining a series of simple functions. Because FuncADL is written independently of the underlying data libraries, it can run on many data formats.
### When to use FuncADL
| | Uproot | FuncADL |
|---|--------|---------|
| **Strengths** | Ideal for working with **ROOT ntuples** or flat data structures.<br>Use preexisting knowledge of Uproot to build queries.<br>Queries run quickly and are easy to set up. | Designed for getting all possible data from **xAOD datasets**.<br>Allows writing queries in **Python syntax** that are translated into optimized C++ and run in AnalysisBase.<br>Anything that can be done in AnalysisBase can be done with FuncADL.<br>Removes need for cloning, changing, and building AnalysisBase. |
| **Limitations** | Limited to simpler transformations and filtering.<br>Does not natively handle complex object hierarchies. | Steeper learning curve; use only when necessary.<br>Runs slower than Uproot ServiceX. |

For most analyses, Uproot queries will suffice. If you are not sure where to start, it is recommended to start there. 
If you frequently work with xAOD file types and need access to values beyond the standard set of objects, it is recommended that you start with FuncADL.

## Notebook Setup

An important note for the imports is that we need to import the FuncADL query function from the specific release that we are working with. Here we will use r25, but r21/r22 are supported, simply replace the number in the import.

In [ ]:
import awkward as ak
import matplotlib.pyplot as plt
from servicex import deliver, dataset
from servicex_analysis_utils import to_awk
from func_adl_servicex_xaodr25 import FuncADLQueryPHYSLITE

## Understanding Query Structure

### Starting our Query
To start each query you must use the `FuncADLQueryPHYSLITE` function to create a base query.

In [ ]:
base_query = FuncADLQueryPHYSLITE()

We now have an object that we can apply the FuncADL operators on to build our query before we send it to ServiceX.

### Using .Select()
ServiceX at its core is designed to deliver specified data back to you for analysis. This requires a way to specify the data that will be returned. For FuncADL that is the `.Select()` operator. We apply the `.Select()` operator onto our query and depending on how many `.Select()` operators there are depends on what the operator does. Below is an example of a query to retrieve jet pt, eta, and phi.

In [ ]:
base_query = FuncADLQueryPHYSLITE()
jets_per_event = (base_query
   .Select(lambda e: e.Jets()) # This select loops over the events and specifies which containers we pass to the next loop
   .Select(lambda jets: { # This select opens those containers
           'pt': jets.Select(lambda j: j.pt() / 1000), # This .Select() grabs all of the values for this attribute and puts them into a list.
           'eta': jets.Select(lambda j: j.eta()),
           'phi': jets.Select(lambda j: j.phi()),
   })
)

Notice the pattern: the **outer**`.Select()` calls step through the event and pick containers, while the **inner**`.Select()` loops over the objects inside a container.
Also notice `j.pt() / 1000`: xAOD files store all energies and momenta in **MeV**, so we divide by 1000 to work in GeV.

### From Query to Data
A query on its own does nothing until we send it to ServiceX. Exactly as in the 15 Minute Challenge, we point at a dataset, package the query into a spec, and call `deliver()`.
Your instructor will provide the dataset name for today's workshop — paste it into the cell below.

In [ ]:
ds = dataset.Rucio("mc20_13TeV:DAOD_PHYSLITE.38191209._000001.pool.root.1")

In [ ]:
spec = {
   'Sample': [{
       'Name': 'Example1_Jets',
       'Dataset': ds,
       'Query': jets_per_event
   }]
}
results = deliver(spec)

In [ ]:
data = to_awk(results)['Example1_Jets']
plt.hist(ak.flatten(data.pt), bins=100, range=(0, 200))
plt.xlabel('Jet $p_T$ [GeV]')
plt.ylabel('Number of jets')
plt.title('Example 1: Jet $p_T$')
plt.show()

>**Try it yourself:**  add a `'mass'` entry to the dictionary in the query above using `j.m()`, rerun the three cells, and plot it. Remember MeV → GeV!

### Using .Where()
The `.Where()` operator applies cuts. It takes a lambda that returns `True` or `False` for each item and keeps only the items that pass.
**The location of `.Where()` in the query determines what gets cut.** Applied to a container, it removes *objects*. The query below keeps only jets with $p_T > 30$ GeV:

In [ ]:
base_query = FuncADLQueryPHYSLITE()
cut_jets_per_event = (base_query
   .Select(lambda e: e.Jets().Where(lambda j: j.pt() / 1000 > 30)) # only jets passing the cut are passed on
   .Select(lambda jets: {
       'pt': jets.Select(lambda j: j.pt() / 1000),
   })
)

In [ ]:
spec = {
   'Sample': [{
       'Name': 'Example2_JetCut',
       'Dataset': ds,
       'Query': cut_jets_per_event
   }]
}
cut_data = to_awk(deliver(spec))['Example2_JetCut']
plt.hist(ak.flatten(data.pt), bins=100, range=(0, 200), label='All jets')
plt.hist(ak.flatten(cut_data.pt), bins=100, range=(0, 200), histtype='step', linewidth=2, label='$p_T > 30$ GeV')
plt.xlabel('Jet $p_T$ [GeV]')
plt.ylabel('Number of jets')
plt.title('Example 2: Object-level cut')
plt.legend()
plt.show()

### Event-Level Cuts (Skimming)
If `.Where()` comes **before the first `.Select()`**, it acts on whole *events*: any event where the lambda returns `False` is dropped entirely.
The query below keeps only events that contain at least one jet above 100 GeV (`.Count()` returns the number of objects in a collection), then returns the jet $p_T$ for the events that survive.

In [ ]:
base_query = FuncADLQueryPHYSLITE()
skimmed_jets = (base_query
   .Where(lambda e: e.Jets().Where(lambda j: j.pt() / 1000 > 1000).Count() > 0) # cuts whole events
   .Select(lambda e: e.Jets())
   .Select(lambda jets: {
       'pt': jets.Select(lambda j: j.pt() / 1000),
   })
)
spec = {
   'Sample': [{
       'Name': 'Example2_Skim',
       'Dataset': ds,
       'Query': skimmed_jets
   }]
}
skim_data = to_awk(deliver(spec))['Example2_Skim']
print(f'Events before skim: {len(data.pt)}')
print(f'Events after skim:  {len(skim_data.pt)}')

## Cheat Sheet
**Physics object collections**available in the first `.Select()`:
`e.Jets()`, `e.Electrons()`, `e.Muons()`, `e.TauJets()`, `e.TrackParticles()`, `e.Vertices()` (and more — see the [FuncADL docs](https://tryservicex.org/funcadl/))
**Common object methods:**`.pt()`, `.eta()`, `.phi()`, `.m()`, `.charge()` (electrons/muons)
- `.Where()` before the first `.Select()` cuts **events**; on a container it cuts **objects**.
- `.Count()` gives the number of objects in a collection.
- Python builtins like `abs()` work inside queries.
**Every ServiceX request follows the same recipe:**1. Build a query
2. Package it in a spec with a dataset and a sample name
3. `deliver()` the spec
4. Load the results with `to_awk()` and analyze

## Problems
Now it's your turn! Work through the problems below — each one follows the recipe from the cheat sheet. Feel free to work in pairs, and flag one of the helpers if you get stuck. Don't worry about finishing everything: solutions will be shared at the end, and there will be time for one-on-one help later today.

### Problem 1: Muons
Build a query that returns the **$p_T$ and $\eta$ of all muons**in each event, deliver it, and plot a histogram of the muon $p_T$ from 0 to 200 GeV.
*Hints: the muon container is `e.Muons()`, and muon momenta are also stored in MeV.*

In [ ]:
base_query = FuncADLQueryPHYSLITE()
muons_per_event = (base_query
   .Select(lambda e: ...)      # 1. pick the muon container
   .Select(lambda muons: {
       # 2. fill in the fields you want
   })
)
spec = {
   'Sample': [{
       'Name': 'Problem1_Muons',
       'Dataset': ds,
       'Query': muons_per_event
   }]
}
muon_data = to_awk(deliver(spec))['Problem1_Muons']
# 3. plot the muon pt

### Problem 2: Central Jets
Build a query that returns the $p_T$ of only the jets in the central part of the detector: $|\eta| < 1.0$. Deliver it and plot the jet $p_T$.
*Hints: use `.Where()` on the jet container, like in Example 2. `abs()` works inside queries.*

In [ ]:
base_query = FuncADLQueryPHYSLITE()
central_jets = (base_query
   # your query here
)
spec = {
   'Sample': [{
       'Name': 'Problem2_CentralJets',
       'Dataset': ds,
       'Query': central_jets
   }]
}
central_data = to_awk(deliver(spec))['Problem2_CentralJets']
# plot the jet pt

### Problem 3: Skim and Select
Build a query that keeps only events with **at least 2 jets with $p_T > 40$ GeV**, and returns the $p_T$ and $\eta$ of the jets in those events. Deliver it, plot the jet $p_T$, and print how many events survived the skim.
*Hints: put a `.Where()` before the first `.Select()`, and use `.Count()` to count the jets passing the cut, like in the skimming example.*

In [ ]:
base_query = FuncADLQueryPHYSLITE()
dijet_events = (base_query
   # your query here
)
spec = {
   'Sample': [{
       'Name': 'Problem3_Dijets',
       'Dataset': ds,
       'Query': dijet_events
   }]
}
dijet_data = to_awk(deliver(spec))['Problem3_Dijets']
# plot the jet pt and print the number of surviving events

## Bonus: Injecting C++ Code Directly
> **For extreme use cases only.** Everything you will normally need is covered by `.Select()` and `.Where()` (plus `getAttribute[]`). Injecting raw C++ is a **last resort** for the rare cases where a value *cannot* be reached functionally — for example an xAOD method that returns its value through a reference argument (like `summaryValue()` on tracks), or one that needs a dedicated selection tool. If you find yourself reaching for this, check with a ServiceX expert first — there is usually a simpler way.
You will not need this today; it is here so you know it exists. FuncADL can inject a custom C++ function into a query through the `.MetaData()` operator, and it runs as if it were written natively in the EventLoop code.
Because `.MetaData()` cannot be used inline like `.Select()`, wiring up a C++ function takes two pieces:
1. a *callable* function that attaches the C++ source to the query via `.MetaData()`, and
2. a dummy Python *stub*, linked to the callable with the `@func_adl_callable` decorator, so the function name can be used inside a query.
The toy example below squares a number in C++. (Yes, `.Select()` could do this — it is only meant to show the mechanics.) First, some extra imports:

In [ ]:
import ast
from typing import Tuple, TypeVar
from func_adl import ObjectStream, func_adl_callable
T = TypeVar("T")

Now the two pieces. In the metadata, `name` must match the Python stub's name, `code` is the C++ source, and `result` names the C++ variable whose value is returned:

In [ ]:
def square_callable(s: ObjectStream[T], a: ast.Call) -> Tuple[ObjectStream[T], ast.Call]:
   # Attaches the C++ source for `square` to the query stream.
   new_s = s.MetaData(
       {
           "metadata_type": "add_cpp_function",
           "name": "square",              # must match the Python stub below
           "code": ["int result = x * x;\n"],
           "result": "result",            # C++ variable holding the return value
           "include_files": [],
           "arguments": ["x"],
           "return_type": "int",
       }
   )
   return new_s, a

@func_adl_callable(square_callable)
def square(x: int) -> int:
   """Return x squared (implemented in C++ on the backend)."""
   ...

With that in place, `square()` can be used inside a query like any other function:

In [ ]:
base_query = FuncADLQueryPHYSLITE()
squared_numbers = (base_query
   .Select(lambda e: {
       'squared': square(2)
   })
)
spec = {
   'Sample': [{
       'Name': 'Bonus_Cpp',
       'Dataset': ds,
       'Query': squared_numbers
   }]
}
cpp_data = to_awk(deliver(spec))['Bonus_Cpp']
cpp_data.squared

Every event comes back with `squared = 4` — the C++ function ran once per event on the backend.
A realistic use case — wrapping `summaryValue()` on tracks, which returns its result through a reference argument and therefore cannot be called functionally — is worked through on the [Using .MetaData() page](https://tryservicex.org/funcadl/xAOD/metadata.html) of the user guide.

## Wrap Up
You now know the core of FuncADL:
- Queries are built by chaining `.Select()` and `.Where()` onto a base query.
- The *position* of an operator decides whether it acts on events, containers, or objects.
- The ServiceX recipe is always the same: query → spec → `deliver()` → `to_awk()`.
FuncADL can do much more than we covered here: reading arbitrary attributes with `getAttribute[]` (for example the jet `EMFrac` you plotted in the 15 Minute Challenge), choosing specific containers, applying calibrations, and handling systematic variations. All of this is covered in the [FuncADL user guide](https://tryservicex.org/funcadl/), and there will be time for one-on-one instruction later today.